In [2]:
import h5py
from itertools import product
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm

# 加载神经反应数据

# 加载行为数据

file_path1 = 'face10156_z_correlations_real_si.h5'
file_path2 = 'con10156_z_correlations_real_si.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_399 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'z_corr_matrix_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
all_z_correlations_real = load_data(file_path1)
all_z_correlations_con = load_data(file_path2)

msub_r_list = []
conr_list = []

for brain_area in range(400):
    z_corr_matrix_real = all_z_correlations_real[brain_area]
    corr_sametrial_stacked = []

    for matrix in z_corr_matrix_real:
        # Extract the diagonal of the submatrices
        submatrix = np.diag(matrix)
        # Append to the list for current brain area
        corr_sametrial_stacked.append(submatrix)

    # Append the list of stacked arrays to the main list
    msub_r_list.append(corr_sametrial_stacked)
    
realpair = np.array(msub_r_list)  # 假设数据保存为.npy文件

for brain_area in range(400):
    z_corr_matrix_con = all_z_correlations_con[brain_area]
    corr_sametrial_stackedcon = []

    for matrix in z_corr_matrix_con:
        # Extract the diagonal of the submatrices
        submatrix = np.diag(matrix)
        # Append to the list for current brain area
        corr_sametrial_stackedcon.append(submatrix)

    # Append the list of stacked arrays to the main list
    conr_list.append(corr_sametrial_stackedcon)
    
conpair = np.array(conr_list)  # 假设数据保存为.npy文件


In [4]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import multiprocessing as mp
from functools import partial
import os

# 启用 Pandas 与 R DataFrame 互操作
pandas2ri.activate()

# 定义处理单个脑区的函数
def process_region(region, realpair, crosspairs):
    import pandas as pd
    import numpy as np
    import os
    from rpy2.robjects import pandas2ri
    import rpy2.robjects as ro
    pandas2ri.activate()

    process_id = os.getpid()
    print(f"Process {process_id} - Processing region {region}")
    
    try:
        # === 1. 构造配对数据，每行表示一个 trial-pair ===
# === 1. 构造数据框 ===
        data_list = []
        for subject in range(46):
            for t in range(24):
                condition = 1 if t < 12 else 2
                stimulus_group = 2 if subject >= 23 else 1
                y_real = realpair[region, subject, t]
                y_pseudo = crosspairs[region, subject, t]
                data_list.append([subject, t, condition, y_real, y_pseudo, stimulus_group])
        
        df_pairs = pd.DataFrame(data_list, columns=["Subject", "Trial", "Condition", "y_real", "y_pseudo", "StimulusGroup"])
        
        # === 2. 分别计算real和pseudo的z-score ===
        df_pairs["z_real"] = df_pairs.groupby("Subject")["y_real"].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else 0
        )
        df_pairs["z_pseudo"] = df_pairs.groupby("Subject")["y_pseudo"].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else 0
        )
        
        # === 3. 分别标记real和pseudo的离群值 ===
        df_pairs["is_real_outlier"] = df_pairs["z_real"].abs() > 3
        df_pairs["is_pseudo_outlier"] = df_pairs["z_pseudo"].abs() > 3
        
        # === 4. 分别创建清洗后的real和pseudo数据 ===
        df_real_clean = df_pairs[~df_pairs["is_real_outlier"]].copy()[["Subject", "Trial", "Condition", "StimulusGroup", "y_real"]]
        df_pseudo_clean = df_pairs[~df_pairs["is_pseudo_outlier"]].copy()[["Subject", "Trial", "Condition", "StimulusGroup", "y_pseudo"]]
        
        # 检查是否都为空
        if df_real_clean.empty or df_pseudo_clean.empty:
            print(f"Process {process_id} - Skipping region {region}: all data removed after cleaning")
            return None
        
        # === 5. 转换为长格式 ===
        df_real_clean = df_real_clean.rename(columns={"y_real": "y"})
        df_real_clean["PairType"] = 1
        df_pseudo_clean = df_pseudo_clean.rename(columns={"y_pseudo": "y"})
        df_pseudo_clean["PairType"] = 0
        
        df_long = pd.concat([df_real_clean, df_pseudo_clean], ignore_index=True)
        df_long = df_long[["Subject", "Condition", "StimulusGroup", "PairType", "y"]]
        
        # 计算真实数据的平均值 (不变的部分)
        df_real_grouped = df_long.groupby(["Subject", "Condition", "StimulusGroup", "PairType"])["y"].mean().reset_index()
        df_real_grouped["Subject"] = df_real_grouped["Subject"].astype(str)
        df_real_grouped["y"] = df_real_grouped["y"] * 10  # 放大避免数值精度问题

        # === 5. 转换为 R dataframe ===
        r_df = pandas2ri.py2rpy(df_real_grouped)
        ro.globalenv["df"] = r_df

        # === 6. R代码：拟合 lmer + permutation (修改部分) ===
        r_code = """
        library(lme4)
        library(dplyr)

        # 准备 Python 函数需要的原始数据
        raw_data <- df
        
        df$PairType <- as.factor(df$PairType)
        df$Condition <- as.factor(df$Condition)
        df$StimulusGroup <- as.factor(df$StimulusGroup)
        model_full <- lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=df, REML=FALSE)

        singular <- isSingular(model_full)
        if (singular) warning("Singular fit detected")

        real_pairtype_coef <- fixef(model_full)["PairType1"]
        N_PERMUTATIONS <- 3000
        perm_pairtype_coefs <- numeric(N_PERMUTATIONS)
        fail_count <- 0
        """
        
        # 将原始数据也传给R
        df_clean_for_r = df_long.copy()
        df_clean_for_r["Subject"] = df_clean_for_r["Subject"].astype(str)
        df_clean_for_r["y"] = df_clean_for_r["y"] * 10  # 与上面保持一致
        r_raw_df = pandas2ri.py2rpy(df_clean_for_r)
        ro.globalenv["raw_data"] = r_raw_df
        
        # 继续R代码，实现先打乱再平均的permutation
        r_code_perm = """
        for (i in 1:N_PERMUTATIONS) {
            # 使用原始数据（trial级别）
            perm_raw <- raw_data %>% as.data.frame()
            
            for (subj in unique(raw_data$Subject)) {
                for (cond in unique(raw_data$Condition)) {
                    for (grp in unique(raw_data$StimulusGroup)) {
                        idx <- which(raw_data$Subject == subj & raw_data$Condition == cond & raw_data$StimulusGroup == grp)
                        if (length(idx) > 1) {
                            perm_raw$PairType[idx] <- sample(raw_data$PairType[idx])
                        }
                    }
                }
            }
    
            # 重新计算打乱后的平均值
            perm_df <- perm_raw %>%
                       group_by(Subject, Condition, StimulusGroup, PairType) %>%
                       summarize(y = mean(y), .groups = 'drop') %>%
                       as.data.frame()
            
            # 确保因子级别正确
            perm_df$PairType <- factor(perm_df$PairType, levels=c("0", "1"))
            perm_df$Condition <- factor(perm_df$Condition)
            perm_df$StimulusGroup <- factor(perm_df$StimulusGroup)
            
            # 拟合模型
            perm_model <- tryCatch(
                lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=perm_df, REML=FALSE),
                error = function(e) { fail_count <<- fail_count + 1; return(NA) }
            )
            if (is.na(perm_model)[1]) next
            perm_pairtype_coefs[i] <- fixef(perm_model)["PairType1"]
        }

        perm_pairtype_coefs <- perm_pairtype_coefs[!is.na(perm_pairtype_coefs)]
        p_value_pairtype_perm <- mean(perm_pairtype_coefs >= real_pairtype_coef)

        list(
            intercept=fixef(model_full)["(Intercept)"],
            pair=fixef(model_full)["PairType1"],
            condition2=fixef(model_full)["Condition2"],
            stimgroup2=fixef(model_full)["StimulusGroup2"],
            p_value_pairtype_perm=p_value_pairtype_perm,
            singular=singular,
            fail_count=fail_count
        )
        """
        
        r_results = ro.r(r_code + r_code_perm)
        results = [region] + list(r_results)
        print(f"Process {process_id} - Completed region {region}")
        return results

    except Exception as e:
        print(f"Process {process_id} - Error processing region {region}: {e}")
        return None


# 主函数 - 并行处理
def run_analysis(realpair, crosspairs):
    # 设置并行处理的核心数
    num_cores = 8
    print(f"Running with {num_cores} cores")
    
    # 创建进程池
    pool = mp.Pool(processes=num_cores)
    
    # 创建偏函数，固定realpair和crosspairs参数
    process_func = partial(process_region, realpair=realpair, crosspairs=crosspairs)
    
    # 并行处理所有脑区
    results = pool.map(process_func, range(400))
    
    # 关闭进程池
    pool.close()
    pool.join()
    
    # 过滤掉None值
    results_list = [r for r in results if r is not None]
    
    # 创建结果DataFrame
    df_results = pd.DataFrame(results_list, columns=[
        "Brain_Region", "Intercept", "Pair_Effect", 
        "Condition_Effect", "StimulusGroup_Effect",
        "PairType_p_perm", "Singular_Fit", "Fail_Count"
    ])
    
    # 保存结果
    
    return df_results

# 使用方法:
results = run_analysis(np.array(realpair), np.array(conpair))

Running with 8 cores
Process 1594142 - Processing region 0
Process 1594143 - Processing region 13
Process 1594144 - Processing region 26
Process 1594145 - Processing region 39


R[write to console]: Loading required package: Matrix



Process 1594147 - Processing region 52


R[write to console]: Loading required package: Matrix

R[write to console]: Loading required package: Matrix



Process 1594148 - Processing region 65


...

Process 1594144 - Completed region 363


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 1594149 - Completed region 389


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 1594142 - Completed region 376


In [5]:
from statsmodels.stats.multitest import multipletests

# 进行 FDR 校正（对所有脑区的 Time_Condition_p 进行校正）
results["PairType_p_perm_FDR"] = multipletests(results["PairType_p_perm"], method="fdr_bh")[1]
significant_regions1 = results[
    (results["PairType_p_perm_FDR"] < 0.05)]

print(significant_regions1)
# 提取符合条件的脑区编号
significant_region_numbers = significant_regions1["Brain_Region"].tolist()

# 输出脑区编号
print(significant_region_numbers)

     Brain_Region               Intercept            Pair_Effect  \
8               8   [-1.3371769252026657]   [0.5968558176836757]   
50             50   [-0.5531422050821763]   [0.3997866817281806]   
73             73   [-1.5035557398370119]  [0.42149635380398603]   
79             79  [-0.02293176505452778]   [0.5115173063375825]   
86             86   [-0.6122249972639371]  [0.49272981464463256]   
91             91   [-0.3164242553529319]   [0.5409037222158544]   
98             98  [-0.44425795069797736]   [0.5445751550877496]   
142           142    [-0.286003018192897]  [0.48752258396812626]   
205           205   [-0.7372629768356017]   [0.5512270576721606]   
207           207   [-1.4030387819457748]   [1.1894945225702027]   
208           208   [-1.6216680707142084]   [1.1806343147331402]   
295           295  [-0.29119346040597294]  [0.32410042156730035]   
348           348   [-0.2041590440184899]    [0.588132954131621]   
349           349  [-0.19718922224450028]   [0.5

In [6]:
# 保存为 CSV 文件
results.to_csv("sifacecon_0526.csv", index=False)

print("Results saved to 'sifacecon_0526.csv'")

Results saved to 'sifacecon_0526.csv'


In [3]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from statsmodels.stats.multitest import multipletests
pandas2ri.activate()

# ====================== 配置 (按实际改) ======================
CSV_PATH   = "Schaefer2018_400Parcels_17Networks_order.csv"
LABEL_IS_ONE_BASED = False

FACE_ARR    = np.array(realpair)          # (400,46,24) face 条件 DSPS, 变量名按你的改
CONTROL_ARR = np.array(conpair)    # (400,46,24) control 条件 DSPS
PQ_CSV      = "sifacecon_0526.csv"   # 该 face/control 分析的 permutation 结果 CSV(全脑)
REGION_INDEX = [142, 79, 348, 349]        # 该分析显著 parcel 的数组下标
OUT_CSV     = "supp_table_facecontrol_si.csv"

# permutation CSV 列名
COL_REGION = "Brain_Region"
COL_P      = "PairType_p_perm"
COL_Q      = "PairType_p_perm_FDR"     # 已对全400校正; 若CSV无此列, 脚本自动重算
# ============================================================


def unwrap(x):
    if isinstance(x, str):
        return float(x.strip("[] ").split()[0])
    if isinstance(x, (list, tuple, np.ndarray)):
        return float(x[0])
    return float(x)


def load_parcel_labels(csv_path):
    df = pd.read_csv(csv_path, sep=None, engine="python")
    df.columns = [c.strip() for c in df.columns]
    lab_col = next(c for c in df.columns if c.lower().replace(" ", "") in
                   ("roilabel", "label", "index"))
    name_col = next(c for c in df.columns if "name" in c.lower())

    def _clean(s):
        s = str(s).strip()
        for pre in ("17Networks_LH_", "17Networks_RH_", "7Networks_LH_", "7Networks_RH_"):
            if s.startswith(pre):
                return s[len(pre):]
        parts = s.split("_")
        if len(parts) >= 2 and parts[0].endswith("Networks") and parts[1] in ("LH", "RH"):
            return "_".join(parts[2:])
        return s

    def _hemi(s):
        s = str(s)
        return "L" if "_LH_" in s else ("R" if "_RH_" in s else "")

    return ({int(r[lab_col]): _clean(r[name_col]) for _, r in df.iterrows()},
            {int(r[lab_col]): _hemi(r[name_col]) for _, r in df.iterrows()})


def _clean_side(df, col):
    z = df.groupby("Subject")[col].transform(
        lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else 0)
    return z.abs() <= 3


def fit_beta_ci(region, face, control, n_subjects=46, n_trials=24, y_scale=10.0):
    """
    模型同 document 10: y ~ Condition + PairType + StimulusGroup + (1|Subject),
    聚合到 (Subject, Condition, StimulusGroup, PairType) 均值, y*10。
    PairType=1 -> face, 0 -> control。返回 β (face − control) 与 Wald 95% CI (原始尺度)。
    """
    rows = []
    for s in range(n_subjects):
        grp = 2 if s >= 23 else 1
        for t in range(n_trials):
            cond = 1 if t < 12 else 2
            rows.append([s, t, cond, grp, face[region, s, t], control[region, s, t]])
    df = pd.DataFrame(rows, columns=["Subject", "Trial", "Condition",
                                     "StimulusGroup", "y_face", "y_ctrl"])

    keep_f = _clean_side(df, "y_face")
    keep_c = _clean_side(df, "y_ctrl")
    a = df[keep_f][["Subject", "Condition", "StimulusGroup", "y_face"]].rename(columns={"y_face": "y"})
    a["PairType"] = 1
    b = df[keep_c][["Subject", "Condition", "StimulusGroup", "y_ctrl"]].rename(columns={"y_ctrl": "y"})
    b["PairType"] = 0
    long = pd.concat([a, b], ignore_index=True)
    if long.empty:
        return np.nan, np.nan, np.nan

    agg = long.groupby(["Subject", "Condition", "StimulusGroup", "PairType"])["y"].mean().reset_index()
    agg["Subject"] = agg["Subject"].astype(str)
    agg["y"] = agg["y"] * y_scale

    ro.globalenv["df"] = pandas2ri.py2rpy(agg)
    r_code = """
    library(lme4)
    df$Condition <- as.factor(df$Condition)
    df$PairType <- as.factor(df$PairType)
    df$StimulusGroup <- as.factor(df$StimulusGroup)
    df$Subject <- as.factor(df$Subject)
    m <- lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=df, REML=FALSE)
    b  <- as.numeric(fixef(m)["PairType1"])
    ci <- confint(m, parm="PairType1", method="Wald")
    list(beta=b, lo=as.numeric(ci[1]), hi=as.numeric(ci[2]))
    """
    try:
        r = ro.r(r_code)
        d = {n: r[i] for i, n in enumerate(list(r.names))}
        return float(d["beta"][0]) / y_scale, float(d["lo"][0]) / y_scale, float(d["hi"][0]) / y_scale
    except Exception as e:
        print(f"region {region} fit failed: {e}")
        return np.nan, np.nan, np.nan


def build_table(region_index, face, control, pq_csv, label_map, hemi_map, one_based=False):
    pq = pd.read_csv(pq_csv)
    pq[COL_REGION] = pq[COL_REGION].apply(unwrap).astype(int)
    pq["_p"] = pq[COL_P].apply(unwrap)
    if COL_Q in pq.columns:
        pq["_q"] = pq[COL_Q].apply(unwrap)
    else:
        m = pq["_p"].notna()
        pq["_q"] = np.nan
        pq.loc[m, "_q"] = multipletests(pq.loc[m, "_p"].values, method="fdr_bh")[1]
    pq = pq.set_index(COL_REGION)

    face = np.asarray(face)
    control = np.asarray(control)

    rows = []
    for idx in region_index:
        csv_label = idx if one_based else idx + 1
        parcel = label_map.get(csv_label, f"idx{idx}?")
        hemi = hemi_map.get(csv_label, "")

# 被试级口径: 分侧 |z|>3 清洗后, 每被试求均值, 再跨被试
        dfc = pd.DataFrame(
            [[s, face[idx, s, t], control[idx, s, t]]
             for s in range(46) for t in range(24)],
            columns=["Subject", "yf", "yc"])

        zf = dfc.groupby("Subject")["yf"].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else 0)
        zc = dfc.groupby("Subject")["yc"].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) > 0 else 0)

        # 每被试清洗后均值 (与模型聚合一致)
        f_subj = dfc[zf.abs() <= 3].groupby("Subject")["yf"].mean()
        c_subj = dfc[zc.abs() <= 3].groupby("Subject")["yc"].mean()

        # 两条件都保留的被试
        common = f_subj.index.intersection(c_subj.index)
        n_obs = len(common)
        fmean = f_subj.loc[common].mean() if n_obs > 0 else np.nan
        cmean = c_subj.loc[common].mean() if n_obs > 0 else np.nan
        beta, cilo, cihi = fit_beta_ci(idx, face, control)

        pval = float(pq.loc[idx, "_p"]) if idx in pq.index else np.nan
        qval = float(pq.loc[idx, "_q"]) if idx in pq.index else np.nan

        rows.append({
            "Parcel": parcel, "Hemi": hemi, "n": n_obs,
            "Face mean (z)": round(fmean, 3) if np.isfinite(fmean) else "",
            "Control mean (z)": round(cmean, 3) if np.isfinite(cmean) else "",
            "β (Face−Control)": round(beta, 3) if np.isfinite(beta) else "",
            "95% CI": f"[{cilo:.3f}, {cihi:.3f}]" if np.isfinite(cilo) else "",
            "p (perm)": f"{pval:.3g}" if np.isfinite(pval) else "",
            "q (FDR)": f"{qval:.3g}" if np.isfinite(qval) else "",
        })
    return pd.DataFrame(rows)


label_map, hemi_map = load_parcel_labels(CSV_PATH)
table = build_table(REGION_INDEX, FACE_ARR, CONTROL_ARR, PQ_CSV,
                    label_map, hemi_map, one_based=LABEL_IS_ONE_BASED)
print(table.to_string(index=False))
table.to_csv(OUT_CSV, index=False)
print(f"\nSaved -> {OUT_CSV}")

R[write to console]: Loading required package: Matrix



           Parcel Hemi  n  Face mean (z)  Control mean (z)  β (Face−Control)         95% CI p (perm) q (FDR)
    ContB_PFCmp_1    L 46          0.043            -0.006             0.049 [0.028, 0.070]        0       0
DorsAttnB_PostC_8    L 46          0.038            -0.013             0.051 [0.027, 0.075]        0       0
    ContB_PFClv_4    R 46          0.026            -0.033             0.059 [0.028, 0.090]        0       0
    ContB_PFCmp_1    R 46          0.037            -0.018             0.055 [0.033, 0.077]        0       0

Saved -> supp_table_facecontrol_si.csv
